# 🏍️ استخراج برداری و امبدینگ تردهای چت تلگرام با BAAI/bge-m3 و Qdrant

این نوت‌بوک برای اجرا در **Google Colab با پردازنده گرافیکی رایگان (GPU T4)** طراحی شده است.

### مراحل کار:
1. اتصال به GPU در کولب (`Runtime -> Change runtime type -> T4 GPU`)
2. نصب پیش‌نیازها (`sentence-transformers` و `qdrant-client`)
3. آپلود فایل `threads.jsonl` (تولید شده توسط سیستم)
4. تولید امبدینگ‌ها با مدل **BAAI/bge-m3** با شتاب‌دهنده سخت‌افزاری GPU
5. ذخیره در دیتابیس برداری **Qdrant** به همراه متادیتا و لینک‌های ارجاع به تلگرام
6. دانلود فایل زیپ `qdrant_db.zip` برای انتقال به سرور نهایی

In [ ]:
# ۱. بررسی فعال بودن کارت گرافیک GPU
!nvidia-smi

In [ ]:
# ۲. نصب پکیج‌های لازم
!pip install -q sentence-transformers qdrant-client tqdm rich

### ۳. آپلود فایل `threads.jsonl`
سلول زیر را اجرا کنید و فایل `threads.jsonl` (حجم حدود ۱۲ مگابایت) را انتخاب کنید:

In [ ]:
from google.colab import files
import os

if not os.path.exists('threads.jsonl'):
    print("لطفاً فایل threads.jsonl را انتخاب و آپلود کنید:")
    uploaded = files.upload()
else:
    print("✓ فایل threads.jsonl قبلاً در محیط موجود است.")

### ۴. لود مدل `BAAI/bge-m3` و اتصال به Qdrant

In [ ]:
import torch
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device used: {device}")

print("در حال بارگذاری مدل BAAI/bge-m3 روی GPU...")
model = SentenceTransformer("BAAI/bge-m3", device=device)

# مقداردهی اولیه Qdrant در حالت لوکال و امبدد (بدون نیاز به سرور مجزا)
COLLECTION_NAME = "motor_threads"
client = QdrantClient(path="./qdrant_db")

if not client.collection_exists(COLLECTION_NAME):
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(size=1024, distance=Distance.COSINE),
    )
print("✓ کالکشن Qdrant با موفقیت آماده شد.")

### ۵. خواندن تردها و امبدینگ دسته‌ای (Batch Encoding)

In [ ]:
import json
from tqdm.auto import tqdm
from qdrant_client.models import PointStruct

# خواندن تمام تردهای موجود در فایل
threads = []
with open("threads.jsonl", "r", encoding="utf-8") as f:
    for idx, line in enumerate(f):
        if line.strip():
            item = json.loads(line)
            item["point_id"] = idx + 1
            threads.append(item)

total_threads = len(threads)
print(f"تعداد کل تردها برای امبدینگ: {total_threads:,}")

# اجرای امبدینگ به صورت دسته‌ای (Batch Size 64 روی GPU)
BATCH_SIZE = 64

for i in tqdm(range(0, total_threads, BATCH_SIZE), desc="امبدینگ در Qdrant"):
    batch = threads[i : i + BATCH_SIZE]
    texts = [item["text"] for item in batch]

    # تولید وکتور با BGE-M3 (نرمالایز شده برای فاصله کسینوسی)
    with torch.no_grad():
        embeddings = model.encode(texts, batch_size=BATCH_SIZE, normalize_embeddings=True, show_progress_bar=False)

    # ساخت پوینت‌های Qdrant به همراه متادیتای کامل
    points = []
    for item, emb in zip(batch, embeddings):
        points.append(
            PointStruct(
                id=item["point_id"],
                vector=emb.tolist(),
                payload={
                    "thread_id": item["thread_id"],
                    "chat_id": item["chat_id"],
                    "root_message_id": item["root_message_id"],
                    "message_ids": item["message_ids"],
                    "senders": item["senders"],
                    "date": item["date"],
                    "text": item["text"],
                    "message_links": item["message_links"],
                },
            )
        )

    client.upsert(collection_name=COLLECTION_NAME, points=points)

print("\n✓ تمام تردها با موفقیت در Qdrant ذخیره شدند!")

### ۶. فشرده‌سازی پوشه دیتابیس Qdrant برای انتقال به سرور

In [ ]:
import os

# بستن کلاینت جهت اطمینان از فلاش شدن کامل همه وکتورها روی دیسک
client.close()

# فشرده‌سازی پوشه qdrant_db در قالب یک فایل زیپ کم‌حجم
!zip -r -q qdrant_db.zip qdrant_db
zip_size_mb = os.path.getsize("qdrant_db.zip") / (1024 * 1024)
print(f"✓ فایل qdrant_db.zip با حجم {zip_size_mb:.2f} مگابایت آماده دانلود شد.")

### ۷. دانلود مستقیم فایل دیتابیس
با اجرای سلول زیر، فایل دیتابیس به صورت مستقیم در سیستم شما دانلود می‌شود تا آن را روی سرور یا کنار پروژه قرار دهید:

In [ ]:
from google.colab import files
files.download("qdrant_db.zip")

### ۸. تست سرچ معنایی (اختیاری)
می‌توانید همین الان در کولب سوال بپرسید و تردهای مرتبط به همراه آیدی پیام‌ها را مشاهده کنید:

In [ ]:
# اتصال مجدد برای تست سرچ در نسخه‌های جدید Qdrant (متد query_points)
test_client = QdrantClient(path="./qdrant_db")
query = "صدای تق تق تعویض دنده گیربکس دومینار"

query_vector = model.encode(query, normalize_embeddings=True).tolist()
response = test_client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_vector,
    limit=2,
)
results = response.points

print(f"🔍 نتایج جستجو برای: '{query}'\n")
for i, hit in enumerate(results, 1):
    print(f"--- نتیجه {i} (امتیاز تشابه: {hit.score:.4f}) ---")
    print(hit.payload["text"])
    print("لینک‌ها:", hit.payload["message_links"])
    print()
